## Normalization


As we have found out, LeWM was trained on normalized image inputs. 
Until know we have ignored this and did not train out models on normalized data.
In this file we want to compare the changes on the model's performance comparing normalized and unnormalized data.
This will be done across 2 experiments with a subset of 2k samples.
- position probing: extract true positon (x,y) from the embedding
- true distance: extract the true distance between the encoded position of an agent and the encoded position of the goal

In [1]:
# correct working directory only once 
if not "working_directory_corrected" in vars():
    %cd ..
    working_directory_corrected = True

/Users/tomseidel/Desktop/CODE/JEPAHeuristics


In [8]:
import os

# macOS-only fix for a torch/xgboost conflict (not needed on the Spark)
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import pandas as pd

from utils.embedding_probing.experiments.true_distance.generate_true_distance_dataset import (
    generate_true_distance_dataset,
)
from utils.embedding_probing.experiments.position_probing.extraction import position_dataset_to_xy
from utils.embedding_probing.experiments.true_distance.extraction import heuristic_dataset_to_xy
from utils.embedding_probing.training import train_and_compare_models

### Shared Dataset generation


Below we will create 2 sorts of dataset for our models.
1. unnormalized data -- using generate_true_distance_dataset with normalize = False
2. normalized data   -- using generate_true_distance_dataset with normalize = True

In [3]:
NUM_SAMPLES = 2000
SEED = 42

dataset_unnormalized = generate_true_distance_dataset(
    num_samples=NUM_SAMPLES, seed=SEED, device="cpu", batch_size=64, normalize=False,
)
dataset_normalized = generate_true_distance_dataset(
    num_samples=NUM_SAMPLES, seed=SEED, device="cpu", batch_size=64, normalize=True,
)

Initializing Environment and Model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 984.06it/s]

17:39:53 | INFO  | utils.py    | Loading checkpoint from folder /Users/tomseidel/.cache/huggingface/hub/models--quentinll--lewm-tworooms/snapshots/77adaae0bc31deab21c93740d1f8bb947cd0bdec...
17:39:53 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
Generating 2000 samples...


Progress: 1600/2000
Progress: 2000/2000
Initializing Environment and Model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 3217.72it/s]

17:40:35 | INFO  | utils.py    | Loading checkpoint from folder /Users/tomseidel/.cache/huggingface/hub/models--quentinll--lewm-tworooms/snapshots/77adaae0bc31deab21c93740d1f8bb947cd0bdec...
17:40:35 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
Generating 2000 samples...


Progress: 1600/2000
Progress: 2000/2000


### Position-Probing

Compares all four models (Linear Regression, Random Forest, XGBoost, MLP) on our standard evaluation metrics, using the same dataset once with normalized and once with unnormalized embeddings. Each model predicts the true (x, y) position from a single JEPA embedding.

In [4]:
X_unnorm, y_unnorm = position_dataset_to_xy(dataset_unnormalized)
X_norm, y_norm = position_dataset_to_xy(dataset_normalized)

print(f"unnormalized: X={X_unnorm.shape}, y={y_unnorm.shape}")
print(f"normalized:   X={X_norm.shape}, y={y_norm.shape}")

unnormalized: X=(4000, 192), y=(4000, 2)
normalized:   X=(4000, 192), y=(4000, 2)


In [ ]:
N_TRIALS = 10 # keep optuna optimization minimal for now 

results_unnorm = train_and_compare_models(
    X_unnorm, y_unnorm, n_trials=N_TRIALS, device="cpu",
    models_dir="models/normalization_check/unnormalized", seed=SEED,
)
results_norm = train_and_compare_models(
    X_norm, y_norm, n_trials=N_TRIALS, device="cpu",
    models_dir="models/normalization_check/normalized", seed=SEED,
)

[I 2026-11-02 17:43:48,285] A new study created in memory with name: no-name-8b6eca0f-1da7-407e-b230-a0a11a7b4810
[I 2026-11-02 17:43:50,085] Trial 0 finished with value: 32.45556616299861 and parameters: {'n_estimators': 218, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: 32.45556616299861.
[I 2026-11-02 17:43:54,139] Trial 1 finished with value: 30.655181472208092 and parameters: {'n_estimators': 440, 'max_depth': 19, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 30.655181472208092.
[I 2026-11-02 17:44:05,499] Trial 2 finished with value: 33.02713981841997 and parameters: {'n_estimators': 132, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 1 with value: 30.655181472208092.
[I 2026-11-02 17:44:06,343] Trial 3 finished with value: 33.37640503936934 and parameters: {'n_estimators': 112, 'max_depth': 11, 'min_

In [6]:
comparison = pd.concat(
    {
        "unnormalized": results_unnorm.set_index("model"),
        "normalized": results_norm.set_index("model"),
    },
    axis=0,
)[["MAE", "R2", "Pearson", "L2Error"]]

comparison

MAE        R2   Pearson    L2Error
             model                                                      
unnormalized Linear Regression  23.691522  0.716869  0.848542  37.123681
             Random Forest      30.561801  0.494856  0.714144  48.547756
             XGBoost            25.187463  0.600523  0.777287  41.591199
             MLP                44.093860  0.128131  0.365137  67.284729
normalized   Linear Regression   4.323013  0.989153  0.994574   6.846374
             Random Forest       5.401989  0.974623  0.988093   8.378012
             XGBoost             3.055237  0.990917  0.995843   4.952176
             MLP                 2.769191  0.990390  0.995362   4.490263

## True-Distance
Compares all four models (Linear Regression, Random Forest, XGBoost, MLP) on our standard evaluation metrics, using the same datasets once with normalized and once with unnormalized embeddings. Each model predicts the true wall-aware distance between the agent and goal state from their paired JEPA embeddings. 


In [9]:
X_unnorm_td, y_unnorm_td = heuristic_dataset_to_xy(dataset_unnormalized)
X_norm_td, y_norm_td = heuristic_dataset_to_xy(dataset_normalized)

print(f"unnormalized: X={X_unnorm_td.shape}, y={y_unnorm_td.shape}")
print(f"normalized:   X={X_norm_td.shape}, y={y_norm_td.shape}")

unnormalized: X=(2000, 384), y=(2000,)
normalized:   X=(2000, 384), y=(2000,)


In [10]:
results_unnorm_td = train_and_compare_models(
    X_unnorm_td, y_unnorm_td, n_trials=N_TRIALS, device="cpu",
    models_dir="models/normalization_check/true_distance_unnormalized", seed=SEED,
)
results_norm_td = train_and_compare_models(
    X_norm_td, y_norm_td, n_trials=N_TRIALS, device="cpu",
    models_dir="models/normalization_check/true_distance_normalized", seed=SEED,
)

[I 2026-11-02 18:07:48,925] A new study created in memory with name: no-name-c7ad4d04-9816-4001-8e64-072469159b76
[I 2026-11-02 18:07:49,918] Trial 0 finished with value: 52.1308542094936 and parameters: {'n_estimators': 218, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: 52.1308542094936.
[I 2026-11-02 18:07:52,411] Trial 1 finished with value: 51.90526016226857 and parameters: {'n_estimators': 440, 'max_depth': 19, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 1 with value: 51.90526016226857.
[I 2026-11-02 18:08:02,146] Trial 2 finished with value: 51.86023377737665 and parameters: {'n_estimators': 132, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 2 with value: 51.86023377737665.
[I 2026-11-02 18:08:02,659] Trial 3 finished with value: 51.63557990066918 and parameters: {'n_estimators': 112, 'max_depth': 11, 'min_sampl

In [11]:
comparison_td = pd.concat(
    {
        "unnormalized": results_unnorm_td.set_index("model"),
        "normalized": results_norm_td.set_index("model"),
    },
    axis=0,
)[["MAE", "R2", "Pearson", "L2Error"]]

comparison_td

MAE        R2   Pearson    L2Error
             model                                                      
unnormalized Linear Regression  59.705385 -0.329652  0.247778  59.705385
             Random Forest      54.095757 -0.008315  0.107154  54.095757
             XGBoost            55.718289 -0.077134  0.068966  55.718289
             MLP                54.180072 -0.050738 -0.060220  54.180072
normalized   Linear Regression  56.325047 -0.145751  0.257612  56.325047
             Random Forest      45.436558  0.283160  0.586276  45.436558
             XGBoost            38.936141  0.393361  0.638369  38.936141
             MLP                15.584594  0.896733  0.947183  15.584594